<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 5 (2): LangGraph — Drawing the Loop

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Understand **state** — the one dictionary that every step reads and writes
2. Build a graph from **nodes** and **edges**, and draw it
3. Let the graph **branch** with a conditional edge — the routing pattern
4. Tell a **workflow** from an **agent**, and know which one you actually need
5. Put an **LLM** inside a node
6. Rebuild notebook 1's agent as a graph with **`ToolNode`** and **`tools_condition`**
7. Give it **memory** with a checkpointer and a `thread_id`
8. **Pause** a run before a tool fires, inspect it, and resume — human in the loop

> **Notebook 1 first.** This one assumes you know what a tool call is and have seen the
> agent loop written by hand. Here we rebuild that loop as something you can draw.

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q langgraph langchain-openai

In [ ]:
import os
from getpass import getpass
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key
MODEL = "gpt-4o-mini"

print("Setup complete")

---

## 2. Why a Graph?

In notebook 1 you wrote this:

```python
for step in range(MAX_ITERATIONS):
    response = client.chat.completions.create(...)
    if response.choices[0].finish_reason != "tool_calls":
        return response.choices[0].message.content
    ...run the tools and go round again
```

It works. But everything about it is invisible: the state lives in local variables, the control flow
lives in an `if`, and if you want to pause halfway you are out of luck.

**LangGraph is that same loop, written as a picture.**

| Your loop | A graph |
|---|---|
| local variables | explicit **state** every node reads and writes |
| an `if` inside a `for` | **nodes** joined by **edges** |
| pausing means rewriting it | **checkpoints** — pause, inspect, resume |
| you print to debug | you **draw** it |

Three words carry the whole library:

```
  STATE   a dictionary that flows through the graph
  NODE    a function: takes the state, returns an update to it
  EDGE    what runs next
```

That's it. Let's build one with no LLM at all, so the mechanics are unmistakable.

---

## 3. Your First Graph

A tiny support-ticket pipeline: classify a ticket, draft a reply, sign it.

**Step 1 — describe the state.** This is the shape of the dictionary that will travel through the
graph. Every node can read all of it and write any part of it.

In [ ]:
class TicketState(TypedDict):
    text: str         # what the customer wrote
    category: str     # filled in by the classifier
    reply: str        # filled in by the drafter


print(TicketState.__annotations__)

**Step 2 — write the nodes.** A node is a plain function. It receives the whole state and
returns a dictionary containing **only the keys it wants to change**.

No LLM here on purpose - a node is just a function.

In [ ]:
def classify(state: TicketState):
    """Decide what kind of ticket this is."""
    text = state["text"].lower()
    if "refund" in text or "invoice" in text or "charged" in text:
        category = "billing"
    elif "error" in text or "crash" in text or "login" in text:
        category = "technical"
    else:
        category = "general"

    print(f"  [node] classify -> {category}")
    return {"category": category}          # only the key it changed


def draft_reply(state: TicketState):
    """Write a reply based on what the classifier decided."""
    print(f"  [node] draft_reply (category = {state['category']})")
    return {"reply": f"Thanks for getting in touch about your {state['category']} query."}


def add_signature(state: TicketState):
    """Tidy up the finished reply."""
    print("  [node] add_signature")
    return {"reply": state["reply"] + "\n\n- The Support Team"}

**Step 3 — wire them together.** Add each node, then say what follows what. `START` and `END`
are the two built-in markers.

In [ ]:
builder = StateGraph(TicketState)

# Nodes: a name, and the function to run
builder.add_node("classify", classify)
builder.add_node("draft_reply", draft_reply)
builder.add_node("add_signature", add_signature)

# Edges: what runs next. A straight line, for now.
builder.add_edge(START, "classify")
builder.add_edge("classify", "draft_reply")
builder.add_edge("draft_reply", "add_signature")
builder.add_edge("add_signature", END)

graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

**Step 4 — run it.** `invoke()` takes the starting state and returns the final state.

In [ ]:
result = graph.invoke({"text": "I was charged twice for my invoice", "category": "", "reply": ""})

print("\ncategory :", result["category"])
print("reply    :", result["reply"])

Look at what came back: **the whole state**, not just the last node's return value.

> Each node returned only the key it changed. LangGraph **merged** those updates into one dictionary
> and carried it forward. That merging is the single most important thing to understand about state.

---

## 4. State — What Actually Flows

Two rules and you know everything about state:

1. A node **receives the whole state**.
2. A node **returns only the keys it wants to change**.

Here is a node that proves both.

In [ ]:
def inspect_state(state: TicketState):
    """Read every key, change exactly one."""
    print("  This node can see the whole state:")
    print(f"     text     = {state['text']!r}")
    print(f"     category = {state['category']!r}")
    print(f"     reply    = {state['reply']!r}")

    return {"category": state["category"].upper()}      # and it only changes this one


# Swap it into the pipeline in place of add_signature
b2 = StateGraph(TicketState)
b2.add_node("classify", classify)
b2.add_node("inspect_state", inspect_state)
b2.add_edge(START, "classify")
b2.add_edge("classify", "inspect_state")
b2.add_edge("inspect_state", END)

out = b2.compile().invoke({"text": "the app keeps crashing on login", "category": "", "reply": ""})
print("\nFinal state:", out)

---

## 5. Conditional Edges — Letting the Graph Branch

So far the path was fixed: classify, then draft, then sign, every single time.

A **conditional edge** lets the graph choose. You write a **router**: a function that looks at the
state and returns *the name of the next node*.

In [ ]:
def route_by_category(state: TicketState) -> str:
    """A router returns the NAME of the next node - just a string."""
    if state["category"] == "billing":
        return "billing_reply"
    elif state["category"] == "technical":
        return "technical_reply"
    else:
        return "general_reply"


def billing_reply(state: TicketState):
    print("  [node] billing_reply")
    return {"reply": "Our billing team will review your invoice within 2 working days."}


def technical_reply(state: TicketState):
    print("  [node] technical_reply")
    return {"reply": "Please try clearing your cache. An engineer will follow up shortly."}


def general_reply(state: TicketState):
    print("  [node] general_reply")
    return {"reply": "Thanks for your message - we will get back to you soon."}

In [ ]:
router_builder = StateGraph(TicketState)

router_builder.add_node("classify", classify)
router_builder.add_node("billing_reply", billing_reply)
router_builder.add_node("technical_reply", technical_reply)
router_builder.add_node("general_reply", general_reply)

router_builder.add_edge(START, "classify")

# The conditional edge: after `classify`, ask the router where to go next
router_builder.add_conditional_edges("classify", route_by_category)

router_builder.add_edge("billing_reply", END)
router_builder.add_edge("technical_reply", END)
router_builder.add_edge("general_reply", END)

router_graph = router_builder.compile()

display(Image(router_graph.get_graph().draw_mermaid_png()))

The dotted lines in that picture are the branch. Now change the ticket text and re-run the
next cell a few times - watch a different path light up each time.

In [ ]:
# Change this line and re-run: try "I was charged twice" / "the app crashes" / "hello there"
ticket = "I keep getting an error when I try to login"

result = router_graph.invoke({"text": ticket, "category": "", "reply": ""})
print("\ncategory :", result["category"])
print("reply    :", result["reply"])

### You just built the routing pattern

That graph is a **workflow**: you decided the possible paths in advance, and a classifier picks one.
Compare it with notebook 1's agent, where the *model* decided what to do next.

| | **Workflow** *(this graph)* | **Agent** *(notebook 1)* |
|---|---|---|
| Who chooses the path | **you**, at write time | **the model**, at run time |
| Possible outcomes | you can list them all | you cannot |
| Testing | straightforward | hard - it varies |
| Cost | predictable | varies per question |

> **If you can write down the steps, write down the steps.** Autonomy is a cost, not a feature -
> every decision you hand to the model is one you can no longer test, price, or explain to a
> customer. Most production systems marketed as "AI agents" are workflows, and they are right to be.

Reach for an agent only when you genuinely **cannot** list the steps in advance.

---

## 6. Putting an LLM in a Node

A node is just a function, so a node that calls an LLM is just a function that calls an LLM.

One new thing: for conversations, the state holds a **list of messages** that we want to **append**
to rather than overwrite. That is what `add_messages` does.

In [ ]:
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL)


class ChatState(TypedDict):
    # Annotated[..., add_messages] means "when a node returns messages, APPEND them"
    messages: Annotated[list, add_messages]


def chatbot(state: ChatState):
    """One LLM call, as a node."""
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
chat_builder = StateGraph(ChatState)
chat_builder.add_node("chatbot", chatbot)
chat_builder.add_edge(START, "chatbot")
chat_builder.add_edge("chatbot", END)

chat_graph = chat_builder.compile()

result = chat_graph.invoke({"messages": [{"role": "user", "content": "What is the capital of India?"}]})
print(result["messages"][-1].content)

> Without `add_messages`, the second node to return `messages` would **overwrite** the first.
> With it, the conversation accumulates. That one annotation is what makes chat work.

---

## 7. Tools in a Graph — the Loop, Drawn

Now we rebuild notebook 1's agent. Same idea, three differences:

| Notebook 1 (by hand) | Here |
|---|---|
| you wrote the JSON schema | the **`@tool`** decorator builds it from your type hints and docstring |
| `handle_tool_calls()` | **`ToolNode`** |
| `if finish_reason == "tool_calls"` | **`tools_condition`** |

The docstring **is** the description the model reads - the same rule as notebook 1.

In [ ]:
from langchain_core.tools import tool
from datetime import datetime


@tool
def get_current_time() -> str:
    """Get the current date and time. Use for anything about 'today', 'now', or dates."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S (%A)")


@tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression such as '250 * 50000'. Use for any calculation."""
    return str(eval(expression))


tools = [get_current_time, calculate]

# Look at what the decorator generated for you
print(calculate.name)
print(calculate.description)
print(calculate.args)

In [ ]:
# bind_tools tells the model which tools exist - the same job the schemas did in notebook 1
llm_with_tools = ChatOpenAI(model=MODEL).bind_tools(tools)


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: AgentState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

agent_builder = StateGraph(AgentState)

agent_builder.add_node("agent", agent_node)
agent_builder.add_node("tools", ToolNode(tools=tools))     # runs whatever the model asked for

agent_builder.add_edge(START, "agent")

# tools_condition is a ready-made router: "tools" if the model asked for one, END if it didn't
agent_builder.add_conditional_edges("agent", tools_condition)

# ...and this edge is what makes it a LOOP - after the tools run, go back to the model
agent_builder.add_edge("tools", "agent")

agent_graph = agent_builder.compile()

display(Image(agent_graph.get_graph().draw_mermaid_png()))

**Look at that picture.** The arrow from `tools` back to `agent` is the cycle - and that cycle
**is** the `while` loop you wrote in notebook 1. Same behaviour, now something you can point at.

In [ ]:
# Needs a tool
result = agent_graph.invoke({"messages": [{"role": "user", "content": "What is 47853 * 1942?"}]})
print(result["messages"][-1].content)

In [ ]:
# Needs nothing - tools_condition sends it straight to END, so the loop never runs
result = agent_graph.invoke({"messages": [{"role": "user", "content": "What is the capital of France?"}]})
print(result["messages"][-1].content)

In [ ]:
# Every step of the run is in the message list - this is your loop trace, for free
result = agent_graph.invoke({"messages": [{"role": "user", "content": "What day is it today?"}]})

print(f"{len(result['messages'])} messages in the trace:")
print([type(m).__name__ for m in result["messages"]])

---

## 8. Memory

Run the graph twice and it remembers nothing - each `invoke()` starts from scratch.

To fix that you add a **checkpointer**: something that saves the state after every step. Then you
label each conversation with a **`thread_id`**, and the graph picks up where that thread left off.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory_graph = agent_builder.compile(checkpointer=InMemorySaver())

# thread_id identifies ONE conversation
config = {"configurable": {"thread_id": "student-42"}}

r = memory_graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Ravi."}]}, config=config)
print(r["messages"][-1].content)

In [ ]:
# Same thread_id -> it remembers
r = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=config)
print(r["messages"][-1].content)

In [ ]:
# Change one string and the memory is gone
other = {"configurable": {"thread_id": "someone-else"}}

r = memory_graph.invoke({"messages": [{"role": "user", "content": "What's my name?"}]}, config=other)
print(r["messages"][-1].content)

⚠️ **That string is what separates one user's conversation from another's.** Hardcode
`thread_id="1"` and ship it, and every user shares one conversation - everyone reading everyone
else's chat. In a real application it comes from your login system, not from a demo.

You can also look at the saved state directly:

In [ ]:
state = memory_graph.get_state(config)

print("Next node to run :", state.next)              # empty tuple = finished
print("Messages stored  :", len(state.values["messages"]))

---

## 9. Human in the Loop

Here is what the checkpointer really buys you.

If the state is saved after every step, you can **stop** the graph before a dangerous node, look at
what it intends to do, and only then let it continue. That is `interrupt_before`.

In [ ]:
paused_graph = agent_builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["tools"],        # stop before ANY tool runs
)

cfg = {"configurable": {"thread_id": "approval-demo"}}

paused_graph.invoke({"messages": [{"role": "user", "content": "What is 999 * 111?"}]}, config=cfg)

print("Graph is paused. Nothing has run yet.")

In [ ]:
# Look at exactly what it wants to do, before it does it
state = paused_graph.get_state(cfg)

print("Paused before  :", state.next)
print("Wants to call  :", [tc["name"] for tc in state.values["messages"][-1].tool_calls])
print("With arguments :", [tc["args"] for tc in state.values["messages"][-1].tool_calls])

In [ ]:
# Approve by resuming with None. To reject, simply never call this.
resumed = paused_graph.invoke(None, config=cfg)

print(resumed["messages"][-1].content)

> **You cannot pause what you cannot save.** That is why the checkpointer exists, and why memory
> and human-in-the-loop are the same feature wearing two hats.

In notebook 1 the approval gate was a Python `if` inside your own tool runner. Here the framework
does it for you. Both put the decision **in your code**, never in the model's prompt - a model can be
argued out of an instruction, but not out of a gate that stops the program.

---

## 10. Exercises

Fill in the blanks (`___`). Hints are in the comments.

### Q1: A fourth branch

Add an `urgent_reply` node to the router graph, and route to it when the ticket text contains
`"urgent"`.

In [ ]:
def urgent_reply(state: TicketState):
    return {"reply": "___"}


def route_v2(state: TicketState) -> str:
    # Hint: a router just returns a node NAME as a string.
    if "urgent" in state["___"].lower():
        return "___"
    return route_by_category(state)


b = StateGraph(TicketState)
b.add_node("classify", classify)
b.add_node("urgent_reply", urgent_reply)
b.add_node("billing_reply", billing_reply)
b.add_node("technical_reply", technical_reply)
b.add_node("general_reply", general_reply)
b.add_edge(START, "classify")
b.add_conditional_edges("classify", ___)
for node in ["urgent_reply", "billing_reply", "technical_reply", "general_reply"]:
    b.add_edge(node, END)

print(b.compile().invoke({"text": "urgent - I was charged twice", "category": "", "reply": ""}))

### Q2: A tool of your own

Write a `@tool` that the model cannot do without, add it to the agent graph, and ask a question that
needs it.

In [ ]:
@tool
def get_employee_count() -> str:
    """___"""                       # remember: the docstring IS the description
    return "250"


my_tools = [get_current_time, calculate, ___]

my_llm = ChatOpenAI(model=MODEL).bind_tools(___)

# Hint: rebuild the graph exactly as in section 7, but with my_tools and my_llm.

### Q3: Two threads

Show that two `thread_id` values keep two separate conversations.

In [ ]:
# Hint: the config shape is {"configurable": {"thread_id": "<any string>"}}
config_a = {"configurable": {"thread_id": "___"}}
config_b = {"configurable": {"thread_id": "___"}}

memory_graph.invoke({"messages": [{"role": "user", "content": "My favourite colour is green."}]}, config=___)

r = memory_graph.invoke({"messages": [{"role": "user", "content": "What is my favourite colour?"}]}, config=___)
print(r["messages"][-1].content)     # which config did you pass? That decides the answer.

### Q4: Reject an action

Using `paused_graph`, start a run, inspect what it wants to do — and then **do not resume it**.
Print the pending tool call and explain in a comment why nothing happened.

In [ ]:
cfg2 = {"configurable": {"thread_id": "___"}}

paused_graph.invoke({"messages": [{"role": "user", "content": "___"}]}, config=cfg2)

pending = paused_graph.get_state(___)
print("Would have called:", [tc["name"] for tc in pending.values["messages"][-1].___])
# Nothing ran because ___

---

## Key Takeaways

1. **Three words: state, node, edge.** State is a dictionary that flows; a node is a function that
   updates part of it; an edge says what runs next.

2. **Nodes return only what they change.** LangGraph merges those partial updates into the state and
   carries it forward.

3. **A conditional edge is a router** — a function that returns the *name* of the next node. That is
   the routing workflow pattern, and it is the cheapest useful thing in this notebook.

4. **Workflow vs agent: who picks the path.** You, at write time, or the model, at run time. If you
   can write down the steps, write down the steps.

5. **`add_messages` means append, not overwrite.** Without it a conversation cannot accumulate.

6. **`ToolNode` + `tools_condition` + one edge back = the agent loop**, drawn as a cycle. Same
   behaviour as notebook 1, now visible.

7. **A checkpointer plus a `thread_id` is memory.** Change the string and the memory changes with it
   — so that string belongs to your auth layer, not to a demo.

8. **`interrupt_before` pauses a run with its state intact**, so a human can look before anything
   irreversible happens. You cannot pause what you cannot save.

### Concept Map

```
            START
              |
              v
        +-----------+   tools_condition says "tools"    +---------+
        |   agent   | --------------------------------> |  tools  |
        | (LLM node)| <-------------------------------- | ToolNode|
        +-----------+       results appended            +---------+
              |
              | tools_condition says END
              v
             END

   state  ---> every node reads it, returns only what it changes
   checkpointer + thread_id ---> memory, and the ability to PAUSE
```

### Quick Reference

| Piece | What it does |
|---|---|
| `TypedDict` | describes the shape of the state |
| `Annotated[list, add_messages]` | append to this key instead of overwriting |
| `StateGraph(State)` | the builder |
| `add_node("name", fn)` | register a step |
| `add_edge(a, b)` | always go from a to b |
| `add_conditional_edges(a, router)` | ask the router which node comes next |
| `START` / `END` | the built-in entry and exit markers |
| `.compile()` | turn the builder into a runnable graph |
| `.invoke(state)` | run it; returns the final state |
| `draw_mermaid_png()` | draw it |
| `@tool` | build the schema from type hints + docstring |
| `bind_tools(tools)` | tell the model which tools exist |
| `ToolNode(tools)` | run whatever the model asked for |
| `tools_condition` | ready-made router: tools, or END |
| `InMemorySaver()` | the checkpointer |
| `thread_id` | which conversation this is |
| `interrupt_before=[...]` | pause before those nodes |

### 🏠 Homework

1. **Draw before you build.** Sketch a graph for a task you care about — nodes, edges, and one
   branch — then implement it and compare your picture with `draw_mermaid_png()`.
2. **Workflow or agent?** Take three tasks from your own life and decide which each one needs. Write
   one sentence of justification for each.
3. **Break the memory.** Build a two-turn conversation, then deliberately pass the wrong `thread_id`
   on the second turn. Explain what a user would experience if you shipped that bug.

### 📚 Resources

- [LangGraph documentation](https://langchain-ai.github.io/langgraph/)
- [LangGraph — persistence and checkpointers](https://langchain-ai.github.io/langgraph/concepts/persistence/)
- [LangGraph — human-in-the-loop](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/)
- [Anthropic — Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)

---

**Next:** notebook 3 goes one level higher again — **CrewAI**, where you describe agents by their
*role* and let a crew divide the work.